In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from jcopml.pipeline import num_pipe, cat_pipe
from jcopml.utils import save_model, load_model
from jcopml.plot import plot_missing_value
from jcopml.feature_importance import mean_score_decrease

In [2]:
df = pd.read_csv("data/Social_media_impact_on_life.csv")
df.head()

,Student_ID,Age,Gender,Academic_Level,Primary_Platform,Daily_Usage_Hours,Weekend_Extra_Hours,Device_Type,Sleep_Duration_Hours,Sleep_Quality_Score,Late_Night_Usage,Social_Comparison_Frequency,Perceived_Stress_Score,Mental_Health_Index,Academic_Performance_GPA,Overall_Impact
0,STU_2024000,19,Female,Undergraduate,Snapchat,7.5,1.5,Smartphone,5.9,2,True,Always,20.0,73,2.73,Neutral
1,STU_2024001,19,Female,Undergraduate,Instagram,4.6,1.3,Smartphone,7.9,3,True,Sometimes,17.0,90,3.21,Beneficial
2,STU_2024002,17,Female,High School,TikTok,10.9,1.7,Smartphone,5.2,1,True,Frequently,18.0,70,3.00,Neutral
3,STU_2024003,16,Female,High School,YouTube,6.0,2.3,Tablet,7.6,4,True,Sometimes,15.0,82,3.42,Beneficial
4,STU_2024004,17,Male,High School,LinkedIn,3.3,2.6,Smartphone,7.6,4,True,Never,16.0,84,3.41,Beneficial


In [3]:
X = df.drop(columns=["Student_ID", "Mental_Health_Index"])
y = df["Mental_Health_Index"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((3600, 14), (900, 14), (3600,), (900,))

In [4]:
preprocessor = ColumnTransformer([
    ('numeric', num_pipe(), ["Age", "Daily_Usage_Hours", "Weekend_Extra_Hours", "Sleep_Duration_Hours", "Perceived_Stress_Score"]),
    ('categoric', cat_pipe(encoder='onehot'), ["Gender","Academic_Level", "Primary_Platform", "Device_Type", "Social_Comparison_Frequency", "Overall_Impact"]),
])

In [5]:
from xgboost import XGBRegressor
pipeline = Pipeline([
    ('prep', preprocessor),
    ('algo', XGBRegressor(n_jobs=-1, random_state=42))
])

In [6]:
from sklearn.model_selection import RandomizedSearchCV
from jcopml.tuning import random_search_params as rsp

model = RandomizedSearchCV(pipeline, rsp.xgb_params, cv=5, scoring='r2', n_iter=50, n_jobs=-1, verbose=1, random_state=42)
model.fit(X_train, y_train)

print(model.best_params_)
print(model.score(X_train, y_train), model.best_score_, model.score(X_test, y_test))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
{'algo__colsample_bytree': np.float64(0.6022918018156072), 'algo__gamma': 4, 'algo__learning_rate': np.float64(0.0533653306637961), 'algo__max_depth': 1, 'algo__n_estimators': 157, 'algo__reg_alpha': np.float64(7.0760220858220135), 'algo__reg_lambda': np.float64(0.010165510266418733), 'algo__subsample': np.float64(0.5486242529461928)}
0.8125722408294678 0.8055018305778503 0.8030905723571777


In [7]:
save_model(model, "kills.pkl")

Model is pickled as model/kills.pkl
